# 商品分析｜商品組合與定價調整

商業問題：  
品類與品牌營收貢獻排名  
分析方法：  
- 彙總各商品類別與品牌的營收貢獻
- 依營收排序，找出前十大商品類別與品牌

In [ ]:
SELECT TOP(10)p.Category AS [商品類別],
       p.Brand AS [品牌],
       round(SUM(s.Order_Value),2) AS [訂單金額]
  FROM india_ecom.dbo.sales AS s
  JOIN india_ecom.dbo.products AS p
    ON s.product_ID=p.product_ID
 GROUP BY p.Category,p.Brand
 ORDER BY [訂單金額] DESC;

(10 個資料列受到影響)

品類          | 品牌       | 訂單金額        
------------+----------+-------------
Electronics | HP       | 951734796.28
Electronics | Noise    | 924086676.76
Electronics | boAt     | 851219349.72
Electronics | Samsung  | 844062171.89
Electronics | Apple    | 780012514.09
Home        | Urban    | 152735242.49
Home        | Godrej   | 151601780.09
Home        | Philips  | 145260169.59
Home        | Prestige | 139198053.32
Home        | IKEA     | 134909457.69
(10 個資料列)

總執行時間: 00:00:03.238

分析結果：  
Electronics 類別完全主導榜單前五名， HP 以近 9.5 億居冠， Noise、boAt、Samsung、Apple 則緊追在後。 Home 類別的品牌則落居第 6 至 10 名，顯示營收貢獻高度集中於 Electronics 類別。

商業問題：  
折扣深度與銷售表現關聯分析  
分析方法：  
- 依折扣幅度劃分折扣區間
- 彙總各折扣區間的商品訂單量
- 比較不同折扣深度的銷售表現

In [3]:
WITH dp AS (SELECT Product_ID,
                   CASE
                   WHEN Discount_Percent<=10 THEN '0-10%'
                   WHEN Discount_Percent<=20 THEN '11-20%'
                   WHEN Discount_Percent<=30 THEN '21-30%'
                   WHEN Discount_Percent<=40 THEN '31-40%'
                   WHEN Discount_Percent<=50 THEN '41-50%'
                   ELSE '50% up'
                   END AS [折扣區間]
                   FROM india_ecom.dbo.products)
SELECT d.[折扣區間],
       SUM(s.Quantity) AS [總訂單量]
  FROM dp AS d
  JOIN india_ecom.dbo.sales AS s
    ON d.Product_ID=s.Product_ID
 GROUP BY d.[折扣區間]
 ORDER BY d.[折扣區間];

(5 個資料列受到影響)

折扣區間   | 總訂單量 
-------+------
0-10%  | 40789
11-20% | 66278
21-30% | 68264
31-40% | 65850
41-50% | 71214
(5 個資料列)

總執行時間: 00:00:08.329

分析結果：  
0-10% 折扣區間的總訂單量明顯低於其他區間，為 40,789 件；折扣達 11% 以上後，各區間的總訂單量約落在 66,000～71,000 件之間，差異相對有限。整體而言，較低折扣區間的訂單量較少，而 11% 以上各折扣區間的訂單量則較為接近。

商業問題：  
商品評價與銷售表現關聯分析  
分析方法：  
- 彙總各商品訂單量與平均評價
- 比較不同評價商品的銷售表現
- 觀察評價與訂單量的關聯

In [ ]:
SELECT s.Product_ID,
       SUM(s.Quantity) AS [訂單量],
       round(p.Avg_Rating,2) AS [平均評價]
  FROM india_ecom.dbo.sales AS s
  JOIN india_ecom.dbo.products AS p
    ON s.Product_ID= p.Product_ID
 GROUP BY s.Product_ID,p.Avg_Rating
 ORDER BY [平均評價] DESC;

(2000 個資料列受到影響)

Product_ID | 訂單量 | 平均評價
-----------+-----+-----
PROD001465 | 75  | 4.76
PROD000509 | 67  | 4.7 
PROD001462 | 87  | 4.7 
PROD000542 | 60  | 4.69
PROD000508 | 69  | 4.69
PROD000625 | 98  | 4.69
PROD001393 | 87  | 4.67
PROD000832 | 91  | 4.67
PROD001418 | 92  | 4.67
PROD001136 | 50  | 4.67
PROD001938 | 123 | 4.67
PROD001838 | 75  | 4.67
PROD001954 | 98  | 4.66
PROD000206 | 151 | 4.65
PROD000659 | 79  | 4.65
PROD000874 | 147 | 4.65
PROD001524 | 73  | 4.65
PROD000693 | 80  | 4.64
PROD001433 | 140 | 4.64
PROD000928 | 71  | 4.63
PROD000062 | 97  | 4.62
PROD000658 | 78  | 4.61
PROD000480 | 133 | 4.61
PROD000663 | 135 | 4.61
PROD001285 | 168 | 4.61
PROD000975 | 75  | 4.61
PROD001669 | 140 | 4.61
PROD001208 | 68  | 4.61
PROD001349 | 83  | 4.61
PROD001698 | 69  | 4.61
PROD001991 | 127 | 4.61
PROD000031 | 137 | 4.6 
PROD000047 | 138 | 4.6 
PROD001245 | 68  | 4.6 
PROD000089 | 77  | 4.59
PROD000022 | 122 | 4.59
PROD000260 | 137 | 4.59
PROD000153 | 159 | 4.59
PROD000622 | 99  | 4.59

分析結果：  
共 2,000 筆商品資料，將商品依平均評價由高至低排序後，可以看到評價相同的商品，其訂單量仍可能存在明顯差異。例如評價同為 4.65 分的商品，訂單量分別為 73、79、147 與 151 件。顯示商品評價與訂單量之間並非單純由評價高低決定，其他商品特徵也可能影響銷售表現。

商業問題：  
庫存去化速度與滯銷風險分析  
分析方法：  
- 整合商品庫存與銷售資料
- 彙總各商品的庫存數量與訂單量
- 比較庫存與銷售表現，找出庫存去化較慢商品

In [6]:
SELECT p.Product_ID,
       p.Stock_Quantity AS [庫存數量],
       ISNULL(SUM(s.Quantity),0) AS [訂單量]
  FROM india_ecom.dbo.products AS p
  LEFT JOIN india_ecom.dbo.sales as s
    ON p.Product_ID=s.Product_ID
 GROUP BY p.Product_ID,p.Stock_Quantity
 ORDER BY [庫存數量] DESC;

(2000 個資料列受到影響)

Product_ID | 庫存數量 | 訂單量
-----------+------+----
PROD000767 | 1000 | 158
PROD001020 | 1000 | 299
PROD001049 | 1000 | 143
PROD000912 | 998  | 267
PROD000691 | 997  | 91 
PROD000762 | 997  | 81 
PROD001230 | 997  | 260
PROD001733 | 997  | 79 
PROD000863 | 996  | 154
PROD001190 | 996  | 144
PROD001362 | 996  | 121
PROD000185 | 995  | 239
PROD000090 | 995  | 75 
PROD000123 | 995  | 62 
PROD000504 | 995  | 131
PROD001961 | 994  | 254
PROD001724 | 993  | 193
PROD001740 | 993  | 128
PROD001056 | 992  | 238
PROD001459 | 992  | 123
PROD000189 | 991  | 143
PROD001893 | 990  | 282
PROD001663 | 989  | 80 
PROD000459 | 989  | 265
PROD001869 | 989  | 122
PROD000573 | 988  | 139
PROD000969 | 988  | 111
PROD001027 | 988  | 136
PROD001978 | 988  | 76 
PROD000431 | 987  | 152
PROD000986 | 987  | 209
PROD001993 | 987  | 165
PROD001197 | 987  | 236
PROD000622 | 986  | 99 
PROD000620 | 985  | 122
PROD001674 | 985  | 78 
PROD001827 | 985  | 234
PROD000766 | 984  | 137
PROD000872 | 983  | 162

分析結果：  
列出 2,000 項商品的庫存數量與對應訂單量後，按庫存數量由高至低排序。從資料中可觀察到，庫存量接近 995-1000 件的商品，訂單量從 62 件到 299 件不等，顯示高庫存並不保證高訂單量。這份清單可作為後續評估庫存去化效率的參考。